# 09. Text Translation — via Azure AI Translator REST API

**This is the second of two "text translation" notebooks in this folder.** Where `04_text_translation.py`
translated text by prompting a general-purpose LLM, this notebook (`09_text_translation.py`) calls the
**dedicated Azure AI Translator REST API** directly (`translator/text/translate`) using plain `requests` calls
— no LLM, no SDK, just HTTP. It translates one source message into **three target languages in a single API
call**, which is a structural capability the LLM-prompt loop in notebook 04 doesn't have.

**Difficulty:** Beginner–Intermediate


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `requests` — plain HTTP calls, no dedicated SDK used in this script
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Language/Translator-capable resource (a multi-service Azure AI Foundry resource, or a dedicated
  Translator resource) with a key

**Environment variables** (add to root `.env`):
```
AZURE_LANGUAGE_ENDPOINT=https://<your-resource>.services.ai.azure.com/
AZURE_LANGUAGE_KEY=<your-resource-key>
```
This notebook reuses the same `AZURE_LANGUAGE_ENDPOINT`/`AZURE_LANGUAGE_KEY` variables as notebooks 05–07,
since the Translator REST path (`translator/text/translate`) is served off the same unified resource endpoint.


## What You'll Learn

- How to call the Azure AI Translator REST API directly with `requests`, without an SDK
- The request body shape: `inputs` array with `Text`, `language` (source), and `targets` (array of target
  language objects)
- How translating to multiple target languages in **one** API call compares to the one-call-per-language LLM
  loop in notebook 04
- How to parse the nested Translator response shape (`result["value"][0]["translations"]`)


### Step 1 — Configuration and the `translate_text()` function

`ENDPOINT`, `API_VERSION`, and `SUBSCRIPTION_KEY` are read from the environment rather than hardcoded.
`translate_text()` builds the REST request: the `Ocp-Apim-Subscription-Key` header for auth (same pattern as
notebook 08's MCP call — it's the universal Cognitive Services header), and a JSON body with an `inputs` array.
Each input item carries the text, its known `language` (source), and a `targets` array — this is what allows
one call to request multiple output languages at once.

💡 **Exam tip:** Azure AI Translator's REST API accepts an explicit `api-version` query parameter, just like
Azure AI Language and most other Cognitive Services REST APIs — AI-102 expects familiarity with reading/
constructing these versioned REST URLs, not just using SDKs that hide the version string from you.

🔄 **Alternatives:** Use the dedicated `azure-ai-translation-text` SDK instead of raw `requests` — it wraps
this same REST surface with typed request/response objects and built-in retry policies, similar to how
`azure-ai-textanalytics` wraps the Language REST API in notebooks 05–07.


In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

ENDPOINT = os.environ["AZURE_LANGUAGE_ENDPOINT"]
API_VERSION = "2025-10-01-preview"
SUBSCRIPTION_KEY = os.environ["AZURE_LANGUAGE_KEY"]


def translate_text(text, targets, source_language):
    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        "Content-Type": "application/json"
    }
    url = f"{ENDPOINT}translator/text/translate?api-version={API_VERSION}"
    body = {
        "inputs": [
            {
                "Text": text,
                "language": source_language,
                "targets": targets
            }
        ]
    }

    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    return response.json()

### Step 2 — Translate one message into French, Japanese, and Spanish in a single call

`targets` is a list of `{"language": <code>}` objects — three target languages requested at once, unlike
notebook 04's Python `for` loop that made one full model round-trip per language. `response.raise_for_status()`
turns any non-2xx HTTP response into a raised exception, caught by the surrounding `try/except` in `main()`.

💡 **Exam tip:** Requesting multiple target languages in a single Translator call is more than a convenience —
it reduces the number of round trips (lower latency, fewer requests against any rate limit) compared to calling
once per language, which is the key structural advantage of using the dedicated Translator service over the
per-language LLM-prompt loop in notebook 04 for high-volume translation workloads.

🔄 **Alternatives:** Omit `source_language` (or pass `None`/omit the `language` key on the input) to let the
Translator service **auto-detect** the source language instead of specifying it — the response then also
tells you what it detected, similar in spirit to how the LLM prompt in notebook 04 was asked to report the
detected source language itself.


In [ ]:
def main():
    text = (
        "Our VPN keeps dropping every 10 minutes since the last update. "
        "This is affecting our whole sales team."
    )
    targets = [
        {"language": "fr"},
        {"language": "ja"},
        {"language": "es"},
    ]
    source_language = "en"

    try:
        result = translate_text(text, targets, source_language)

        for t in result["value"][0]["translations"]:
            print(f"[{t['language']}] {t['text']}")

    except Exception as e:
        print(f"Translation failed: {e}")


main()

## Summary

This notebook called the Azure AI Translator REST API directly with `requests`, translating one message into
three target languages (French, Japanese, Spanish) in a single HTTP round trip. Compared to notebook 04's
LLM-prompted, one-call-per-language approach, this is the "dedicated service" side of the translation
trade-off: fewer round trips for multi-language output, certified language coverage, and no prompt to
maintain — at the cost of losing the LLM's ability to adapt tone/register beyond what the MT model itself
produces.


## Try It Yourself

1. **Easy:** Add `"de"` (German) and `"zh-Hans"` (Simplified Chinese) to the `targets` list and rerun.
2. **Intermediate:** Remove `source_language` from the call (omit the `"language"` key from the input dict) and
   inspect the raw JSON response to see how the service reports the auto-detected source language.
3. **Advanced:** Rewrite `translate_text()` using the `azure-ai-translation-text` SDK instead of raw
   `requests`, and compare the resulting code for readability, error handling, and retry behavior against this
   REST version.
